> ℹ️ **About this notebook**
>
> - This is a rewritten version of [../02_king_county_project.ipynb](../02_king_county_project.ipynb).
> - The code is identical. Only the paths are adjusted: `data/` became `../data/`, and `assets/` became `../assets/`.
> - Each step has its own code cell with `# Step` comments for your work.
> - The original has no `solutions/` folder. A solution path in German is in [../summary/loesungsweg.ipynb](../summary/loesungsweg.ipynb).
> - You can set the font and font size in your editor.

# King County House Prices Project

Notebook 2 of 2 · Reading time: approx. 360 minutes incl. exercises

## 🎯 What you will learn

1. You split the data into a **training set** and a **test set** before you explore it.
2. You build a **pipeline** with a **ColumnTransformer** and tune two models with **cross-validation**.
3. You compare every model with two **baselines** and find the largest errors of the best model.

## 🗺️ Roadmap

| Section | Topic | Exercise? |
| :--- | :--- | :--- |
| 1️⃣ Section 1 of 6 | The project in one pass | no |
| 2️⃣ Section 2 of 6 | The data | no |
| 3️⃣ Section 3 of 6 | Goal, data and split | ✏️ Exercise 1, 2 and 3 |
| 4️⃣ Section 4 of 6 | Explore and build baselines | ✏️ Exercise 4 and 5 |
| 5️⃣ Section 5 of 6 | Pipeline and tuning | ✏️ Exercise 6 and 7 |
| 6️⃣ Section 6 of 6 | Evaluate on the test set | ✏️ Exercise 8 |

## 📖 New terms

| Term | In one sentence |
| :--- | :--- |
| **regression** | A **regression** predicts a number, here the sale price of a house. |
| **target** | The **target** is the column you predict, here `price`. |
| **feature** | A **feature** is a column that the model uses to predict the **target**. |
| **metric** | A **metric** is a number that tells you how good the predictions are. |
| **RMSE** | The **RMSE** is the typical prediction error in dollars, and large errors count more. |
| **MAE** | The **MAE** is the mean absolute prediction error in dollars. |
| **R²** | The **R²** is the share of the price differences that the model explains. |
| **training set** | The **training set** is the part of the data that you explore and train on. |
| **test set** | The **test set** is the locked part of the data for the final evaluation. |
| **correlation** | The **correlation** measures from minus 1 to plus 1 how closely two columns rise or fall together along a straight line. |
| **skewed** | A column is **skewed** if a few very large values pull its distribution to one side. |
| **missing value** | A **missing value** is an empty cell, shown as `NaN` in pandas. |
| **baseline** | A **baseline** is a simple model that your real models have to beat. |
| **mean baseline** | The **mean baseline** predicts the mean price of the **training set** for every house. |
| **rule-based baseline** | The **rule-based baseline** predicts the price with one simple rule from one column. |
| **pipeline** | A **pipeline** chains preprocessing steps and a model into one object. |
| **ColumnTransformer** | A **ColumnTransformer** applies different preprocessing steps to different columns. |
| **imputing** | **Imputing** fills **missing values**, for example with the median of the column. |
| **scaling** | **Scaling** brings every numeric column to mean 0 and standard deviation 1. |
| **one-hot encoding** | **One-hot encoding** turns a category column into one column of 0 and 1 per category. |
| **regularisation** | **Regularisation** keeps the coefficients of a linear model small, so that the model stays simple. |
| **Ridge** | **Ridge** is a linear **regression** with **regularisation**, and `alpha` sets its strength. |
| **hyperparameter** | A **hyperparameter** is a setting you choose before training, for example `alpha` of **Ridge**. |
| **polynomial features** | **Polynomial features** add the squares and products of the numeric columns. |
| **cross-validation** | **Cross-validation** splits the **training set** into **folds** and scores the model on each **fold** in turn. |
| **fold** | A **fold** is one of the equal parts of the **training set** in **cross-validation**. |
| **random search** | A **random search** tries random **hyperparameter** values, here with `RandomizedSearchCV`. |
| **grid search** | A **grid search** tries every value from a fixed list, here with `GridSearchCV`. |
| **overfitting** | **Overfitting** means that the model fits the **training set** much better than new data. |
| **residual** | A **residual** is the true price minus the predicted price. |

## 1️⃣ Section 1 of 6: The project in one pass

- Your task is to predict the sale price of a house in King County.
- This is a **regression**.
- You also tune the model with **cross-validation**.
- You combine the methods you already know into one complete analysis.

The steps follow a machine learning project in a single pass:

![The project from top to bottom. Step 1 is the goal and the metric, step 2 gets the data, step 3 splits it into training data and locked test data. Steps 4 and 5 explore the data and build baselines on the training data only. Steps 6 and 7 form a loop of feature engineering, setting hyperparameters, training models and calculating the cross-validation score. In step 8 the test data is unlocked and every model is scored on it.](../assets/king_county_project_workflow.png)

👀 What to look for:

- The **test set** stays locked until step 8.
- Steps 6 and 7 form a loop. You repeat them until the cross-validated score stops improving.

## 2️⃣ Section 2 of 6: The data

King County covers Seattle and its surroundings. The data comes as two tables:

- `../data/king_county_house_sales.csv`: one row per sale, with the date and the price.
- `../data/king_county_house_details.csv`: one row per house, with its physical attributes.

`01_loading_the_data.ipynb` combines them into one table.

| Column | Table | Description |
| :-- | :-- | :-- |
| `date` | sales | date of the sale |
| `price` | sales | sale price in dollars, the **target** |
| `house_id` | sales | identifier of the house, matches `id` in the details table |
| `id` | sales | identifier of the sale |
| `id` | details | identifier of the house, once per house |
| `bedrooms` | details | number of bedrooms |
| `bathrooms` | details | number of bathrooms |
| `sqft_living` | details | living area in square feet |
| `sqft_lot` | details | lot area in square feet |
| `floors` | details | number of floors |
| `waterfront` | details | whether the house has a view to a waterfront |
| `view` | details | quality of the view |
| `condition` | details | overall condition of the house |
| `grade` | details | overall grade of the house in the King County grading system |
| `sqft_above` | details | living area without the basement, in square feet |
| `sqft_basement` | details | basement area in square feet |
| `yr_built` | details | year of construction |
| `yr_renovated` | details | year of renovation |
| `zipcode` | details | zip code |
| `lat` | details | latitude |
| `long` | details | longitude |
| `sqft_living15` | details | living area of the 15 nearest neighbours, in square feet |
| `sqft_lot15` | details | lot area of the 15 nearest neighbours, in square feet |

👀 What to look for:

- `price` is the **target**. Most other columns are possible **features**.
- After the merge in notebook 01, the `id` of the details table is called `id_details`.
- The sales run from 2014-05-02 to 2015-05-27.

## 3️⃣ Section 3 of 6: Goal, data and split

Run the import cell first. It loads every library you need for this project.

In [ ]:
# Import the libraries you need for this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

### ✏️ Exercise 1 of 8: Set your goal

1. Choose a **metric** to judge your model.
2. Write down why you chose it.
3. Use the cell "Your answer" below.

🎯 You are done when you named one **metric** and one reason.

The imports offer three **metrics**. Here, $y_i$ is the true price, $\hat{y}_i$ the predicted price, and $n$ the number of houses.

$$\text{RMSE} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2}$$

In words: you square every error, take the mean and then the square root.

$$\text{MAE} = \frac{1}{n}\sum_{i=1}^{n}|y_i - \hat{y}_i|$$

In words: you take the mean of the errors without their signs.

$$R^2 = 1 - \frac{\sum_i (y_i - \hat{y}_i)^2}{\sum_i (y_i - \bar{y})^2}$$

In words: the **R²** compares your squared errors with those of always predicting the mean price $\bar{y}$.

<details>
<summary>💡 Hint</summary>

The functions are `root_mean_squared_error`, `mean_absolute_error` and `r2_score`. Think about which errors are most expensive in a house sale.

</details>

<details>
<summary>✅ Solution</summary>

- One possible choice is the **RMSE**.
- It is in dollars, like the price, so you can read it directly.
- It weights large errors more than small ones.
- Report the **MAE** and the **R²** as well. They help you read the **RMSE**.

</details>

**Your answer:** ...

### ✏️ Exercise 2 of 8: Get the data

1. Complete `01_loading_the_data.ipynb` first. It saves `../data/eda.csv`.
2. Load `../data/eda.csv` into a DataFrame `df`.
3. Check the shape of `df`.

🎯 You are done when `df` has 21,597 rows and 23 columns.

<details>
<summary>💡 Hint</summary>

Use `pd.read_csv()` and the attribute `.shape`.

</details>

<details>
<summary>✅ Solution</summary>

```python
df = pd.read_csv("../data/eda.csv")
df.shape
```

</details>

In [ ]:
# Your code here
# Step 1: Load ../data/eda.csv into df.
# Step 2: Show df.shape.

### ✏️ Exercise 3 of 8: Split the data into a training and a test set

1. Call `train_test_split` once on the whole `df`.
2. Choose the size of the **test set** with `test_size`.
3. Fix `random_state`. Then you get the same split every time.
4. Keep **features** and **target** together in `df_train` and `df_test`.
5. Do not look at `df_test` until Exercise 8.

You do not need a separate validation set. In Exercise 7, **cross-validation** on the **training set** takes its place.

🎯 You are done when `df_train` and `df_test` both still contain the column `price`.

<details>
<summary>💡 Hint</summary>

`train_test_split(df, ...)` with one DataFrame returns two DataFrames.

</details>

<details>
<summary>✅ Solution</summary>

```python
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(df_train.shape, df_test.shape)
```

With `test_size=0.2`, the **training set** has 17,277 rows and the **test set** has 4,320 rows.

</details>

In [ ]:
# Step 1: Split df with train_test_split, test_size and random_state.
# Step 2: Print the shapes of df_train and df_test.

## 4️⃣ Section 4 of 6: Explore and build baselines

> ⚠️ **Important: keep a record of every change you make to the training set by hand.**
>
> - In Exercise 8 you apply exactly the same changes to the **test set**.
> - One example is a new column such as the age of the house at sale.
> - Another example is dropping the identifiers `id` and `house_id` and the column `date`.
> - Steps inside your **pipeline** are applied to the **test set** automatically.
> - These steps are **imputing**, **scaling** and **one-hot encoding**. They use the values learned on the **training set**.

### ✏️ Exercise 4 of 8: Explore the training set

1. Which columns are related to the price? Which **correlation** is strongest?
2. Which columns are **skewed**?
3. Which columns have **missing values**?
4. Which **features** contain the same information?
5. Write your answers into the cell "Your answers".

🎯 You are done when you can answer all four questions with numbers.

<details>
<summary>💡 Hint</summary>

Useful methods: `.corr(numeric_only=True)`, `.skew(numeric_only=True)` and `.isna().sum()`. Plots with `sns.histplot` and `sns.scatterplot` help as well.

</details>

<details>
<summary>✅ Solution</summary>

```python
# Correlation of every numeric column with the price
df_train.corr(numeric_only=True)["price"].sort_values(ascending=False)

# Skewness: large positive values mean a long tail to the right
df_train.skew(numeric_only=True).sort_values(ascending=False)

# Missing values per column
df_train.isna().sum()

# Same information: living area = area above ground + basement
known = df_train.dropna(subset=["sqft_basement"])
(known["sqft_living"] == known["sqft_above"] + known["sqft_basement"]).all()
```

| Question | Result on the **training set** |
| :--- | :--- |
| Strongest **correlation** with the price | `sqft_living` 0.70, then `grade` 0.67, `sqft_above` 0.60 and `sqft_living15` 0.59 |
| Most **skewed** columns | `sqft_lot` 13.37, `waterfront` 11.28, `sqft_lot15` 8.90, `price` 3.95 |
| **Missing values** | `yr_renovated` 3,069 or 17.8 %, `waterfront` 1,907 or 11.0 %, `sqft_basement` 356 or 2.1 %, `view` 53 or 0.3 % |
| Same information | `sqft_living` is exactly `sqft_above` plus `sqft_basement`, and `id_details` repeats `house_id` |

- `sqft_living` and `sqft_above` have a **correlation** of 0.88.
- In `yr_renovated`, 78.8 % of the values are 0. That means "never renovated".

One way to record your changes by hand is a function:

```python
def prepare(data):
    # All changes by hand, in one place, for training and test set
    data = data.copy()
    sale_year = pd.to_datetime(data["date"]).dt.year
    data["age_at_sale"] = sale_year - data["yr_built"]
    data["renovated"] = (data["yr_renovated"] > 0).astype(int)
    return data.drop(columns=["id", "house_id", "id_details", "date",
                              "yr_renovated", "sqft_basement"])
```

- `age_at_sale` lies between -1 and 115 years. The value -1 means a house was sold before its year of construction.
- `renovated` is 1 for 3.4 % of the **training set**.

</details>

In [ ]:
# Step 1: Show the correlation of every numeric column with price.
# Step 2: Show the skewness of every numeric column.
# Step 3: Count the missing values per column.
# Step 4: Look for columns with the same information.
# Step 5: Write your changes by hand into one function.

**Your answers:**

1. ...
2. ...
3. ...
4. ...

### ✏️ Exercise 5 of 8: Separate features and target, and build two baselines

1. Split `df_train` into `X_train` and `y_train`.
2. Leave the **test set** untouched until Exercise 8.
3. Build a **mean baseline** with [`DummyRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html).
4. Build a **rule-based baseline** from the column with the strongest **correlation**.
5. Learn the values of the rule on the **training set** only.

🎯 You are done when both **baselines** predict a price for every house in `X_train`.

<details>
<summary>💡 Hint</summary>

A simple rule is a price per square foot. Calculate it from `y_train` and `X_train['sqft_living']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
train = prepare(df_train)
X_train = train.drop(columns="price")
y_train = train["price"]

# Mean baseline: the mean price of the training set for every house
mean_baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)

# Rule-based baseline: price per square foot, learned on the training set
price_per_sqft = (y_train / X_train["sqft_living"]).median()

def rule_baseline(X):
    return price_per_sqft * X["sqft_living"]
```

The rule learned on the **training set** is a median price of 244.44 dollars per square foot.

$$\hat{y} = 244.44 \cdot \text{sqft\_living}$$

In words: the predicted price is 244.44 dollars times the living area in square feet.

| **Baseline** | Training **RMSE** in dollars |
| :--- | ---: |
| **mean baseline** | 368,958 |
| **rule-based baseline** | 267,068 |

</details>

In [ ]:
# Step 1: Apply your changes by hand to df_train.
# Step 2: Split it into X_train and y_train.
# Step 3: Fit a DummyRegressor as mean baseline.
# Step 4: Learn the value of your rule on the training set.

☕ Halfway there. Take a ten-minute break before you build the **pipeline**.

## 5️⃣ Section 5 of 6: Pipeline and tuning

### ✏️ Exercise 6 of 8: Build a preprocessing pipeline and train a linear model

1. Choose the numeric and the categorical columns.
2. Treat `zipcode` as a category, although it is stored as a number.
3. Build a **ColumnTransformer** with **imputing**, **scaling** and **one-hot encoding**.
4. Put the **ColumnTransformer** and `Ridge` into one **pipeline**.
5. Use as many **features** as possible.
6. Train the **pipeline** on `X_train` and `y_train`.

🎯 You are done when your **pipeline** trains on `X_train` without errors.

<details>
<summary>💡 Hint</summary>

The **pipeline** learns medians and means only from the data you pass to `.fit()`. Use `OneHotEncoder(handle_unknown="ignore")` for zip codes that are missing in the **training set**.

</details>

<details>
<summary>✅ Solution</summary>

```python
cat_cols = ["zipcode"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])
cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
preprocess = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("cat", cat_pipe, cat_cols),
])
ridge = Pipeline([("preprocess", preprocess), ("model", Ridge())])
ridge.fit(X_train, y_train)
```

- `zipcode` has 70 values in the **training set**. **One-hot encoding** turns them into 70 columns.
- After preprocessing, the model sees 87 columns.

</details>

In [ ]:
# Step 1: List the numeric and the categorical columns.
# Step 2: Build a ColumnTransformer with imputing, scaling and one-hot encoding.
# Step 3: Put the ColumnTransformer and Ridge into one Pipeline.
# Step 4: Fit the Pipeline on X_train and y_train.

### ✏️ Exercise 7 of 8: Tune and compare models with cross-validation

1. Define a `KFold` with 5 **folds**, `shuffle=True` and a fixed `random_state`.
2. Tune `alpha` of **Ridge** with a wide **random search**.
3. Narrow it down with a **grid search** around the best value.
4. Repeat steps 2 and 3 for **Ridge** with **polynomial features**.
5. Choose the best model by its cross-validated score.
6. Save every model in its own variable. Also keep the value of your rule.

🎯 You are done when you have two tuned models and their cross-validated **RMSE**.

<details>
<summary>💡 Hint</summary>

Use `scoring="neg_root_mean_squared_error"`. scikit-learn always maximises a score, so it reports the error with a minus sign. Put a minus in front of `best_score_` to get the **RMSE**. `loguniform(1e-4, 1e4)` draws values across many powers of ten.

</details>

<details>
<summary>✅ Solution</summary>

```python
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Wide random search over many powers of ten
search = RandomizedSearchCV(
    ridge, {"model__alpha": loguniform(1e-4, 1e4)}, n_iter=30, cv=cv,
    scoring="neg_root_mean_squared_error", random_state=42,
).fit(X_train, y_train)
best = search.best_params_["model__alpha"]

# Narrow grid search around the best value
grid = GridSearchCV(
    ridge, {"model__alpha": np.geomspace(best / 5, best * 5, 11)}, cv=cv,
    scoring="neg_root_mean_squared_error",
).fit(X_train, y_train)
best_ridge = grid.best_estimator_
print(grid.best_params_, -grid.best_score_)
```

```python
num_poly = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("poly", PolynomialFeatures(degree=2, include_bias=False)),
    ("scale", StandardScaler()),
])
preprocess_poly = ColumnTransformer([
    ("num", num_poly, num_cols),
    ("cat", cat_pipe, cat_cols),
])
ridge_poly = Pipeline([("preprocess", preprocess_poly), ("model", Ridge())])

search_poly = RandomizedSearchCV(
    ridge_poly, {"model__alpha": loguniform(1e-4, 1e4)}, n_iter=30, cv=cv,
    scoring="neg_root_mean_squared_error", random_state=42,
).fit(X_train, y_train)
best = search_poly.best_params_["model__alpha"]
grid_poly = GridSearchCV(
    ridge_poly, {"model__alpha": np.geomspace(best / 5, best * 5, 11)}, cv=cv,
    scoring="neg_root_mean_squared_error",
).fit(X_train, y_train)
best_poly = grid_poly.best_estimator_
print(grid_poly.best_params_, -grid_poly.best_score_)
```

| Model | Columns after preprocessing | Best `alpha` | Cross-validated **RMSE** in dollars |
| :--- | ---: | ---: | ---: |
| **Ridge** | 87 | any value up to 0.3 | ≈ 161,500 |
| **Ridge** with **polynomial features** | 240 | 0.00055 | 132,572 |

- **Ridge** with **polynomial features** has the lower cross-validated **RMSE**. So it is the best model.
- For **Ridge**, every `alpha` up to 0.3 gives almost the same **RMSE**. So the best `alpha` can change from run to run.
- For **Ridge** with **polynomial features**, the best `alpha` lies inside the grid, not at its edge.

</details>

👀 What to look for: if the best `alpha` lies at the edge of your grid, widen the grid and search again.

In [ ]:
# Step 1: Define a KFold with 5 folds.
# Step 2: Tune Ridge with RandomizedSearchCV, then with GridSearchCV.
# Step 3: Tune Ridge with PolynomialFeatures the same way.
# Step 4: Save the best model of each search in its own variable.

## 6️⃣ Section 6 of 6: Evaluate on the test set

### ✏️ Exercise 8 of 8: Prepare the test set and evaluate all models

1. Apply your changes by hand to `df_test`.
2. Split it into `X_test` and `y_test`.
3. Calculate the training and test scores of every model, including both **baselines**.
4. Collect all scores in one table, with one row per model.
5. Which models beat the **baselines**? How large is the gap between training and test scores?
6. Analyse the **residuals** of your best model. For which houses is it wrong, and how large are the errors?

🎯 You are done when your table is complete and you described the largest errors.

<details>
<summary>💡 Hint</summary>

Use the same function for `df_test` as for `df_train`. `pd.qcut(y_test, 4)` splits the test houses into four price quarters.

</details>

<details>
<summary>✅ Solution</summary>

```python
test = prepare(df_test)
X_test = test.drop(columns="price")
y_test = test["price"]

models = {
    "Mean baseline": mean_baseline.predict,
    "Rule baseline": rule_baseline,
    "Ridge": best_ridge.predict,
    "Ridge polynomial": best_poly.predict,
}
rows = []
for name, predict in models.items():
    rows.append({
        "model": name,
        "train_rmse": root_mean_squared_error(y_train, predict(X_train)),
        "test_rmse": root_mean_squared_error(y_test, predict(X_test)),
        "train_mae": mean_absolute_error(y_train, predict(X_train)),
        "test_mae": mean_absolute_error(y_test, predict(X_test)),
        "train_r2": r2_score(y_train, predict(X_train)),
        "test_r2": r2_score(y_test, predict(X_test)),
    })
scores = pd.DataFrame(rows).set_index("model").round(3)
scores
```

All errors are in dollars:

| Model | Train **RMSE** | Test **RMSE** | Train **MAE** | Test **MAE** | Train **R²** | Test **R²** |
| :--- | ---: | ---: | ---: | ---: | ---: | ---: |
| **mean baseline** | 368,958 | 360,907 | 235,058 | 231,542 | 0.00 | 0.00 |
| **rule-based baseline** | 267,068 | 260,103 | 168,642 | 165,076 | 0.48 | 0.48 |
| **Ridge** | ≈ 160,200 | ≈ 161,700 | ≈ 95,600 | ≈ 96,300 | 0.81 | 0.80 |
| **Ridge** with **polynomial features** | 125,303 | 123,490 | 75,848 | 75,765 | 0.88 | 0.88 |

The **Ridge** values are rounded to hundreds, because they change by a few dollars between runs.

- Both **Ridge** models beat both **baselines** on the **test set**.
- Training and test scores are close for every model. So there is no sign of **overfitting**.

```python
residuals = y_test - best_poly.predict(X_test)
print(residuals.describe().round(0))

# Mean absolute residual per price quarter
print(residuals.abs().groupby(pd.qcut(y_test, 4)).mean().round(0))

# Share of test houses with an error above 200,000 dollars
print((residuals.abs() > 200_000).mean().round(3))
```

$$e_i = y_i - \hat{y}_i$$

In words: a **residual** is the true price minus the predicted price, so a positive value means the model predicted too low.

| Price quarter of the **test set** | Mean absolute **residual** in dollars |
| :--- | ---: |
| up to 319,800 | 44,972 |
| up to 450,000 | 52,729 |
| up to 637,500 | 68,239 |
| up to 7,060,000 | 137,264 |

- The errors grow with the price. In the most expensive quarter they are about 3 times as large as in the cheapest.
- 298 test houses or 6.9 % have an error above 200,000 dollars.
- The largest error is a house with 7,120 square feet. It sold for 900,000 dollars, and the model predicted 2,726,355 dollars.

</details>

In [ ]:
# Step 1: Apply your changes by hand to df_test.
# Step 2: Split it into X_test and y_test.
# Step 3: Collect training and test scores of every model in one table.
# Step 4: Calculate and plot the residuals of your best model.

**Your conclusions:** ...

## 📌 Remember

1. Split off the **test set** first and do not look at it until the end.
2. Every model has to beat the **mean baseline** and the **rule-based baseline**.
3. A **pipeline** learns all preprocessing values on the **training set** only.
4. Choose models and **hyperparameters** by **cross-validation**, never by the **test set**.
5. The **residuals** show for which houses the model is wrong.

## ✅ Mini-check

**Question 1 of 2:** Which models beat both **baselines** on the **test set**?

<details>
<summary>Answer</summary>

Both **Ridge** models. Their test **RMSE** is about 161,700 and 123,490 dollars. The **rule-based baseline** reaches 260,103 dollars, and the **mean baseline** 360,907 dollars.

</details>

**Question 2 of 2:** How large is the difference between the training and test scores of the best model?

<details>
<summary>Answer</summary>

Small. **Ridge** with **polynomial features** has a training **RMSE** of 125,303 and a test **RMSE** of 123,490 dollars. The test error is even slightly lower, so there is no sign of **overfitting**.

</details>

➕ Extra questions: optional. They check everything in this notebook.

➕ Extra question 1 of 5: What is the difference between the **training set** and the **test set** in this project?

<details>
<summary>Answer</summary>

You explore, build and tune every model on the **training set** with 17,277 rows. The **test set** with 4,320 rows stays locked until the final evaluation in Exercise 8.

</details>

➕ Extra question 2 of 5: The **mean baseline** has a test **R²** of about 0. What does that mean?

<details>
<summary>Answer</summary>

It explains none of the price differences. That is expected, because the **R²** compares a model with always predicting the mean.

</details>

➕ Extra question 3 of 5: Why do you choose `alpha` by **cross-validation** and not by the **test set**?

<details>
<summary>Answer</summary>

If you choose with the **test set**, it no longer measures how the model does on new data. **Cross-validation** uses only the **training set**, so the **test set** stays a fair final check.

</details>

➕ Extra question 4 of 5: What does `OneHotEncoder(handle_unknown="ignore")` do in the **pipeline**?

<details>
<summary>Answer</summary>

It turns `zipcode` into one column of 0 and 1 per zip code. A zip code that is not in the **training set** gets 0 in every column instead of an error.

</details>

➕ Extra question 5 of 5: The mean absolute **residual** is 44,972 dollars in the cheapest quarter and 137,264 dollars in the most expensive. What does this tell you?

<details>
<summary>Answer</summary>

The model is much less precise for expensive houses. Their errors are about 3 times as large.

</details>

---

➡️ This is the last notebook. Next: the summary in [../summary/zusammenfassung.ipynb](../summary/zusammenfassung.ipynb)

📚 Glossary of all terms: [../summary/zusammenfassung.ipynb#glossar](../summary/zusammenfassung.ipynb#glossar)